# 🚀 Architecture Explorer: Matrix Multiplication
*Compare how Compilers and Hardware Architecture impact DGEMM performance.*

**Instructions:**
1. Run the **Source Code** cell to save the kernel.
2. Run the **Benchmark** cell to see the performance results.

In [1]:
#@title 🚀 Run Benchmark { display-mode: "form" }

#@markdown Configure your hardware experiment below:
matrix_size = 1024 #@param [512, 1024, 2048] {type:"raw"}
optimization_level = "-O3" #@param ["-O0", "-O1", "-O2", "-O3"]
use_native_march = True #@param {type:"boolean"}

import subprocess

# 1. Generate the C code
c_code = f"""
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

void dgemm (size_t n, double* A, double* B, double* C) {{
    for (size_t i = 0; i < n; ++i)
        for (size_t j = 0; j < n; ++j) {{
            double cij = C[i+j*n]; 
            for(size_t k = 0; k < n; k++)
                cij += A[i+k*n] * B[k+j*n]; 
            C[i+j*n] = cij; 
        }}
}}

int main() {{
    const int n = {matrix_size};
    double *a = (double *)malloc(n * n * sizeof(double));
    double *b = (double *)malloc(n * n * sizeof(double));
    double *c = (double *)malloc(n * n * sizeof(double));

    for(int i=0; i<n*n; i++) {{ a[i]=1.0; b[i]=2.0; c[i]=0.0; }}

    struct timespec start, end;
    clock_gettime(CLOCK_MONOTONIC, &start);
    dgemm(n, c, a, b);
    clock_gettime(CLOCK_MONOTONIC, &end);

    double time = (end.tv_sec - start.tv_sec) + (end.tv_nsec - start.tv_nsec) / 1e9;
    printf("Matrix Size: %d | Time: %.4f seconds\\n", n, time);

    free(a); free(b); free(c);
    return 0;
}}
"""

with open("dgemm.c", "w") as f:
    f.write(c_code)

# 2. Build the compiler command
cmd = ["gcc", optimization_level, "dgemm.c", "-o", "benchmark"]
if use_native_march:
    cmd.append("-march=native")

# 3. Compile and Run
print(f"🔨 Compiling with: {' '.join(cmd)}...")
compile_res = subprocess.run(cmd, capture_output=True, text=True)

if compile_res.returncode == 0:
    print("✅ Compilation Successful. Running...")
    run_res = subprocess.run(["./benchmark"], capture_output=True, text=True)
    print(run_res.stdout)
else:
    print("❌ Error during compilation:")
    print(compile_res.stderr)

🔨 Compiling with: gcc -O3 dgemm.c -o benchmark -march=native...
✅ Compilation Successful. Running...
Matrix Size: 1024 | Time: 7.4868 seconds

